In [7]:
from pathlib import Path
import pandas as pd

carpeta  = Path(r'source\caracteristicas_del_establecimiento')

dfs = []

for archivo in sorted(carpeta.glob('*.csv')):
    df = pd.read_csv(archivo, sep=";",encoding='utf-8')

    df.columns = df.columns.str.strip().str.lower()

    df["anio"] = int(archivo.name[:4])

    dfs.append(df)

In [8]:
# Paso 3: comparar columnas entre archivos

# Armamos un diccionario con las columnas de cada archivo
columnas_por_archivo = {}
for f in sorted(carpeta.glob("*.csv")):
    df = pd.read_csv(f, sep=None, engine="python", encoding="utf-8", encoding_errors="replace")
    df.columns = (df.columns
                  .str.strip()
                  .str.lower()
                  .str.replace("\ufeff", "",regex=False)
                  )
    columnas_por_archivo[f.name[:4]] = set(df.columns)

# Tomamos el primer año como referencia
anio_ref = list(columnas_por_archivo.keys())[0]
cols_ref = columnas_por_archivo[anio_ref]

print(f"Referencia: {anio_ref} → {sorted(cols_ref)}\n")

for anio, cols in columnas_por_archivo.items():
    if anio == anio_ref:
        continue
    faltantes = cols_ref - cols
    extras    = cols - cols_ref
    if not faltantes and not extras:
        print(f"✔  {anio}  → igual")
    else:
        print(f"✘  {anio}")
        if faltantes: print(f"     Faltan:  {sorted(faltantes)}")
        if extras:    print(f"     Sobran:  {sorted(extras)}")

Referencia: 2011 → ['ambito', 'bibliotecadisponedealmenosunasi', 'bibliotecafuncionaesespacioexclusivosi', 'cooperadoraconpersoneríajurídica', 'cooperadorano', 'cooperadorasinpersoneríajurídica', 'departamento', 'disponedesalaolaboratoriodeinformáticasi', 'electricidadgeneradoreólico', 'electricidadgeneradorhidráulico', 'electricidadgrupoelectrógeno', 'electricidadotro', 'electricidadpanelfotovoltaicosolar', 'electricidadredpública', 'electricidadsi', 'equipamientobibliotecaequipamientobibliotecawebcam', 'equipamientobibliotecaequipoemisorderadioamfm', 'equipamientobibliotecaequiporeceptorderadioamfm', 'equipamientobibliotecaimpresora', 'equipamientobibliotecareproductordecd', 'equipamientobibliotecareproductordedvd', 'equipamientobibliotecascanner', 'equipamientobibliotecaservidorparausoescolar', 'equipamientobibliotecasistemamultimediaocañón', 'equipamientobibliotecatelevisor', 'equipamientoestablecimientoequipodesonido', 'equipamientoestablecimientoequipoemisorderadioamfm', 'equipam

In [ ]:
dfs = []
for f in sorted(carpeta.glob("*.csv")):
    
    # cargar
    df = pd.read_csv(f, sep=None, engine="python", encoding="utf-8")
    
    # normalizar + eliminar BOM
    df.columns = (df.columns
                  .str.strip()
                  .str.lower()
                  .str.replace("\ufeff", "", regex=False))
    
    # extraer año del nombre del archivo
    df["anio"] = int(f.name[:4])
    
    if "internettipodeserviciogratuitoestado" in df.columns and \
       "internettipodeserviciogratuitootro" in df.columns:
        df["internettipodeserviciogratuito"] = (
            df["internettipodeserviciogratuitoestado"] +
            df["internettipodeserviciogratuitootro"]
        )
        df = df.drop(columns=["internettipodeserviciogratuitoestado", 
                               "internettipodeserviciogratuitootro"])
    
    dfs.append(df)

# concatenar
df_final = pd.concat(dfs, ignore_index=True)

# anio primera columna
cols = ["anio"] + [c for c in df_final.columns if c != "anio"]
df_final = df_final[cols]

df_final.to_csv(r"data\caracteristicas_del_establecimiento_final.csv", index=False, encoding="utf-8-sig")

print("Archivo exportado!")

print(df_final.shape)
print(df_final.head())

Archivo exportado!
(16804, 66)
   anio     provincia departamento   sector  ambito  localizacion  \
0  2011  Buenos Aires   25 DE MAYO  Estatal   Rural            50   
1  2011  Buenos Aires   25 DE MAYO  Estatal  Urbano            41   
2  2011  Buenos Aires   25 DE MAYO  Privado  Urbano            12   
3  2011  Buenos Aires   9 DE JULIO  Estatal   Rural            57   
4  2011  Buenos Aires   9 DE JULIO  Estatal  Urbano            42   

  electricidadsi electricidadredpública electricidadgrupoelectrógeno  \
0             46                     41                                
1             29                     29                                
2              5                      5                                
3             46                     44                                
4             32                     31                            1   

  electricidadpanelfotovoltaicosolar  ...  \
0                                     ...   
1                              